# 07b. Random Forest Hyperparameter Tuning - Automobile Loan Default Prediction

Tunes the Random Forest and compares it with the untuned baseline. Both are evaluated with the same stratified 5-fold cross-validation on the training split, at the default 0.5 threshold. The held-out test rows are discarded at load time, and no operating threshold is chosen in this notebook.


## 1. Setup


In [ ]:
import os
import sys

project_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

from src.config import RANDOM_STATE, TARGET
from src.data.data_cleaning import clean_and_split
from src.evaluation.metrics import evaluate_model, log_result_to_csv
from src.models.random_forest import build_random_forest_pipeline

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

train_df, _ = clean_and_split()  # held-out test rows are discarded here
X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results_dir = "experiments/results" if os.path.exists("experiments/results") else "../experiments/results"
experiments_path = os.path.join(results_dir, "experiments.csv")

print("X_train shape:", X_train.shape)
print("Default rate (train):", round(y_train.mean(), 4))


## 2. Untuned Baseline (same folds)

The baseline is re-evaluated here under the same folds as the tuned model, so the comparison is fair. It uses the untuned Random Forest from `src/models/random_forest.py` at the default 0.5 threshold.


In [ ]:
baseline_pipeline = build_random_forest_pipeline()

baseline_results = evaluate_model(baseline_pipeline, X_train, y_train, model_name="random_forest", cv=5)

for key, value in baseline_results.items():
    print(f"{key}: {value}")

log_result_to_csv(baseline_results, experiments_path)


## 3. Search Space

Each list includes the baseline value, so the untuned configuration is reachable by the search.

- `n_estimators`: number of trees (baseline 300).
- `max_depth`: tree depth limit (baseline `None`, unlimited).
- `min_samples_leaf`: minimum applicants per leaf (baseline 1).
- `max_features`: features considered per split (baseline `sqrt`).
- `class_weight`: class weighting (baseline `balanced`).


In [ ]:
param_distributions = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 8, 12, 16, 20],
    "model__min_samples_leaf": [1, 5, 10, 20, 50],
    "model__max_features": ["sqrt", "log2", 0.3, 0.5],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
}


## 4. Randomized Search

30 sampled configurations, each evaluated with the same stratified 5-fold split and scored by ROC-AUC. Each fold refits the full pipeline, including the feature-selection step, so the search cannot leak information from validation folds. Expect 35-45 minutes.


In [ ]:
search = RandomizedSearchCV(
    baseline_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="roc_auc",
    cv=skf,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best CV ROC-AUC:", round(search.best_score_, 4))
print("Best parameters:", search.best_params_)


## 5. Tuned Model (same folds)


In [ ]:
tuned_results = evaluate_model(
    search.best_estimator_,
    X_train,
    y_train,
    model_name="random_forest_tuned",
    cv=5,
)

for key, value in tuned_results.items():
    print(f"{key}: {value}")

log_result_to_csv(tuned_results, experiments_path)


## 6. Baseline vs Tuned

The tuned model counts as better only if its ROC-AUC gain is larger than the baseline's ROC-AUC standard deviation across folds.


In [ ]:
comparison = pd.DataFrame([baseline_results, tuned_results]).drop(columns=['confusion_matrix'])
comparison = comparison[
    ['model_name', 'roc_auc_mean', 'roc_auc_std', 'pr_auc_mean', 'pr_auc_std',
     'recall_mean', 'precision_mean', 'f1_mean']
]

roc_gain = tuned_results['roc_auc_mean'] - baseline_results['roc_auc_mean']
print(comparison.to_string(index=False))
print()
print(f"ROC-AUC gain: {roc_gain:.4f}  (baseline std: {baseline_results['roc_auc_std']:.4f})")
print("Tuned is clearly better" if roc_gain > baseline_results['roc_auc_std'] else "No clear improvement over baseline")


## 7. Save Search Results and Best Parameters


In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
cv_results.to_csv(os.path.join(results_dir, "random_forest_random_search.csv"), index=False)

best_params_df = pd.DataFrame(
    [{"parameter": k, "value": v} for k, v in search.best_params_.items()]
)
best_params_df.to_csv(os.path.join(results_dir, "random_forest_best_parameters.csv"), index=False)
print(best_params_df.to_string(index=False))


**Finding:** to be filled after execution.

**Decision:** to be filled after execution.
